# Demonstration Data Visualization

Load an HDF5 demo file from `data/` and plot joint positions over time to
verify demonstration quality before training an imitation-learning policy.

In [ ]:
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np

## 1. Load a demo file

Point `DEMO_PATH` to any `.h5` file saved by `scripts/collect_data.py`.

In [ ]:
DATA_DIR = Path("..") / "data"

# List available demos
demos = sorted(DATA_DIR.glob("*.h5"))
print(f"Found {len(demos)} demo(s):")
for d in demos:
    print(f"  {d.name}")

# Select the most recent demo (or set manually)
DEMO_PATH = demos[-1] if demos else None
print(f"\nUsing: {DEMO_PATH}")

In [ ]:
# Load HDF5 data
with h5py.File(str(DEMO_PATH), "r") as f:
    qpos = f["observations/qpos"][:]
    qvel = f["observations/qvel"][:]
    ctrl = f["actions/ctrl"][:]
    n_steps = f["metadata"].attrs["n_steps"]
    created = f["metadata"].attrs["created_at"]

print(f"Demo created: {created}")
print(f"Steps: {n_steps}")
print(f"qpos shape: {qpos.shape}")
print(f"qvel shape: {qvel.shape}")
print(f"ctrl shape: {ctrl.shape}")

## 2. Joint positions over time

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(16, 10), sharex=True)
fig.suptitle("Joint positions (qpos) over time", fontsize=14)

# LEAP hand has 16 actuated joints — plot each one
n_joints = min(qpos.shape[1], 16)
timesteps = np.arange(qpos.shape[0])

joint_names = [
    "if_mcp", "if_rot", "if_pip", "if_dip",
    "mf_mcp", "mf_rot", "mf_pip", "mf_dip",
    "rf_mcp", "rf_rot", "rf_pip", "rf_dip",
    "th_cmc", "th_axl", "th_mcp", "th_ipl",
]

for i, ax in enumerate(axes.flat):
    if i >= n_joints:
        ax.set_visible(False)
        continue
    ax.plot(timesteps, qpos[:, i], linewidth=0.8)
    ax.set_title(joint_names[i] if i < len(joint_names) else f"q{i}",
                 fontsize=9)
    ax.set_ylabel("rad")
    ax.grid(True, alpha=0.3)

axes[-1, 0].set_xlabel("step")
axes[-1, 1].set_xlabel("step")
plt.tight_layout()
plt.show()

## 3. Control signals (actions) over time

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(16, 10), sharex=True)
fig.suptitle("Control signals (ctrl) over time", fontsize=14)

n_ctrl = min(ctrl.shape[1], 16)

for i, ax in enumerate(axes.flat):
    if i >= n_ctrl:
        ax.set_visible(False)
        continue
    ax.plot(timesteps, ctrl[:, i], linewidth=0.8, color="tab:orange")
    ax.set_title(joint_names[i] if i < len(joint_names) else f"ctrl{i}",
                 fontsize=9)
    ax.set_ylabel("rad")
    ax.grid(True, alpha=0.3)

axes[-1, 0].set_xlabel("step")
axes[-1, 1].set_xlabel("step")
plt.tight_layout()
plt.show()

## 4. Joint velocities (sanity check)

Large velocity spikes indicate tracking glitches — filter parameters may
need tuning if this happens frequently.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.set_title("Max |qvel| per timestep")
ax.plot(timesteps, np.max(np.abs(qvel), axis=1), linewidth=0.6,
        color="tab:red", alpha=0.7)
ax.set_xlabel("step")
ax.set_ylabel("|qvel|_max (rad/s)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()